# 🏠 House Price Prediction — End-to-End Data Science Project
### Covers: EDA, Descriptive Stats, Outlier Handling, Normalization, 4 Algorithms, Hyperparameter Tuning, Model Comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/house_data.csv')
print('Shape:', df.shape)
df.head()

## 2. Descriptive Statistics

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
print('Data Types:\n', df.dtypes)
print('\nMissing Values:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

## 3. Data Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Feature Distributions', fontsize=15, fontweight='bold')
cols = ['price', 'sqft_living', 'bedrooms', 'bathrooms', 'grade', 'condition']
for ax, col in zip(axes.flat, cols):
    sns.histplot(df[col], ax=ax, kde=True, color='steelblue')
    ax.set_title(col)
plt.tight_layout()
plt.savefig('../data/distributions.png', dpi=120)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/correlation.png', dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='sqft_living', y='price', hue='grade', palette='viridis', alpha=0.6, ax=axes[0])
axes[0].set_title('Price vs Living Area (colored by Grade)')
sns.boxplot(data=df, x='bedrooms', y='price', palette='Set2', ax=axes[1])
axes[1].set_title('Price Distribution by Bedrooms')
plt.tight_layout()
plt.savefig('../data/scatter_box.png', dpi=120)
plt.show()

## 4. Outlier Detection & Handling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=df['price'], ax=axes[0], color='tomato')
axes[0].set_title('Price — Before Outlier Removal')

Q1, Q3 = df['price'].quantile(0.25), df['price'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df[(df['price'] >= Q1 - 1.5*IQR) & (df['price'] <= Q3 + 1.5*IQR)].copy()
print(f'Removed {len(df) - len(df_clean)} outliers | Remaining: {len(df_clean)}')

sns.boxplot(y=df_clean['price'], ax=axes[1], color='mediumseagreen')
axes[1].set_title('Price — After Outlier Removal')
plt.tight_layout()
plt.savefig('../data/outliers.png', dpi=120)
plt.show()

## 5. Data Wrangling & Feature Engineering

In [ ]:
df_clean['house_age']      = 2024 - df_clean['yr_built']
df_clean['price_per_sqft'] = df_clean['price'] / df_clean['sqft_living']
df_clean['total_rooms']    = df_clean['bedrooms'] + df_clean['bathrooms']
print('New features added: house_age, price_per_sqft, total_rooms')
df_clean[['house_age','price_per_sqft','total_rooms']].describe()

## 6. Data Preprocessing & Normalization

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FEATURES = ['bedrooms','bathrooms','sqft_living','sqft_lot','floors',
            'waterfront','view','condition','grade','house_age','total_rooms']
X = df_clean[FEATURES]
y = df_clean['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

# Visualize normalization effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.DataFrame(X_train, columns=FEATURES).boxplot(ax=axes[0], rot=45)
axes[0].set_title('Before Normalization')
pd.DataFrame(X_train_sc, columns=FEATURES).boxplot(ax=axes[1], rot=45)
axes[1].set_title('After StandardScaler')
plt.tight_layout()
plt.savefig('../data/normalization.png', dpi=120)
plt.show()

## 7. Model Training — 4 Algorithms

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

models = {
    'Linear Regression':  LinearRegression(),
    'Decision Tree':      DecisionTreeRegressor(random_state=42),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    cv    = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2').mean()
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv}
    print(f'{name:25s} | MAE={mae:>10,.0f} | RMSE={rmse:>10,.0f} | R2={r2:.4f} | CV_R2={cv:.4f}')

## 8. Hyperparameter Tuning (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5],
}
grid = GridSearchCV(RandomForestRegressor(random_state=42),
                    param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1)
grid.fit(X_train_sc, y_train)
print('Best Params:', grid.best_params_)
print('Best CV R2:', round(grid.best_score_, 4))

best_rf = grid.best_estimator_
preds_t = best_rf.predict(X_test_sc)
r2_t    = r2_score(y_test, preds_t)
mae_t   = mean_absolute_error(y_test, preds_t)
rmse_t  = np.sqrt(mean_squared_error(y_test, preds_t))
results['RF Tuned'] = {'MAE': mae_t, 'RMSE': rmse_t, 'R2': r2_t, 'CV_R2': r2_t}
print(f'Tuned RF → MAE={mae_t:,.0f} | RMSE={rmse_t:,.0f} | R2={r2_t:.4f}')

## 9. Model Comparison Visualization

In [ ]:
res_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print(res_df.round(4))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')
colors = ['#2ed573','#ffa502','#ff6b81','#1e90ff','#a29bfe']

res_df['R2'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('R² Score (higher = better)')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=30)

res_df['MAE'].plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('MAE (lower = better)')
axes[1].tick_params(axis='x', rotation=30)

res_df['RMSE'].plot(kind='bar', ax=axes[2], color=colors, edgecolor='white')
axes[2].set_title('RMSE (lower = better)')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=120)
plt.show()

## 10. Feature Importance

In [ ]:
fi = pd.Series(best_rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fi.plot(kind='barh', figsize=(10, 6), color='steelblue', edgecolor='white')
plt.title('Feature Importance — Tuned Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120)
plt.show()

## 11. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds_t, alpha=0.5, color='steelblue', edgecolors='white', linewidths=0.3)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title(f'Actual vs Predicted — R²={r2_t:.4f}', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../data/actual_vs_predicted.png', dpi=120)
plt.show()